# Environment Setup

Layer 0 turns raw anamnesis transcripts into standardized epidemiological signal. The pipeline runs in four deterministic-friendly stages: zero shot symptom extraction with a small local LLM constrained to a closed catalog, confidence based human in the loop validation, Bayesian condition scoring against the probabilistic knowledge base, and aggregation into the contract table consumed by the Layer 1 forecasting model.

The extraction engine is dual path. On a GPU runtime it uses Qwen3-4B-Instruct-2507 quantized to four bits. On CPU only runtimes it falls back to a transparent keyword baseline, which also serves as the comparison baseline in the evaluation section. A small Whisper based audio demo closes the loop from speech to structured output.

## Drive Mount and Base Path

In [1]:
"""Mount Google Drive when running on Colab. Outside Colab the notebook falls
back to a local directory so the full pipeline can be tested end to end."""

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    """Base path untuk Google Drive storage"""
    base_path = '/content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton'
    IN_COLAB = True
except ModuleNotFoundError:
    base_path = os.environ.get(
        'MATERNALINK_DATA_ROOT',
        os.path.abspath(os.path.join(os.getcwd(), '..', 'drive_local')),
    )
    IN_COLAB = False

DATA_DIRS = {
    name: os.path.join(base_path, 'data', name)
    for name in [
        'master', 'transactional', 'layer0_output',
        'layer1_mock', 'layer2_output', 'audio_demo', 'eval',
    ]
}
for path in DATA_DIRS.values():
    os.makedirs(path, exist_ok=True)

print(f'Base path: {base_path}')

Mounted at /content/drive
Base path: /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton


## Runtime Detection and Dependencies

The LLM path requires a CUDA device. When the notebook runs on Colab the required packages are installed on the fly. The environment variable MATERNALINK_FORCE_BASELINE can force the keyword baseline even on GPU, which is useful for quick smoke tests.

In [2]:
"""Detect the runtime, install Colab dependencies, and choose the extractor path."""

import subprocess
import sys

if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'transformers>=4.51', 'accelerate', 'bitsandbytes', 'faster-whisper', 'gTTS'],
        check=True,
    )

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except ModuleNotFoundError:
    torch = None
    HAS_CUDA = False

USE_LLM = HAS_CUDA and os.environ.get('MATERNALINK_FORCE_BASELINE', '') != '1'
LLM_MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'

if HAS_CUDA:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Extractor path: {"LLM " + LLM_MODEL_ID if USE_LLM else "keyword baseline (CPU)"}')

GPU: Tesla T4
VRAM: 15.6 GB
Extractor path: LLM Qwen/Qwen3-4B-Instruct-2507


# Load Inputs

## Anamnesis Corpus and Knowledge Base

All inputs come from the simulator notebook: the anamnesis transcripts, the symptom to condition knowledge base with explicit likelihood semantics, the condition priors, facility attributes, and the manual diagnosis stream that the anamnesis signal will complement.

In [3]:
"""Load simulator outputs from the shared data directory."""

import json

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)


def read_csv(folder, filename, parse_period=True):
    df = pd.read_csv(os.path.join(DATA_DIRS[folder], filename))
    if parse_period and 'period' in df.columns:
        df['period'] = pd.to_datetime(df['period'])
    return df


facilities = read_csv('master', 'facilities.csv')
conditions = read_csv('master', 'conditions.csv')
kb_symptom_condition = read_csv('master', 'kb_symptom_condition.csv')
diagnoses_monthly = read_csv('transactional', 'diagnoses_monthly.csv')
anamnesis_records = read_csv('transactional', 'anamnesis_records.csv')
anamnesis_ground_truth = read_csv('transactional', 'anamnesis_ground_truth.csv')

SYMPTOM_NAMES = (
    kb_symptom_condition[['symptom_id', 'symptom_name']]
    .drop_duplicates()
    .set_index('symptom_id')['symptom_name']
    .to_dict()
)
VALID_SYMPTOM_IDS = set(SYMPTOM_NAMES)
HAS_LAB = facilities.set_index('facility_id')['has_lab'].to_dict()

print(f'{len(anamnesis_records)} anamnesis records, {len(VALID_SYMPTOM_IDS)} catalog symptoms')

813 anamnesis records, 15 catalog symptoms


# Symptom Extraction Engine

## Closed Vocabulary and Prompt Design

The extractor never invents labels: the prompt embeds the full catalog with short lexical cues and demands strict JSON. Anything outside the catalog is dropped at parse time, so hallucinated identifiers cannot enter the pipeline.

In [4]:
"""Closed symptom vocabulary with lexical cues and the extraction prompt."""

SYMPTOM_CUES = {
    'G01': 'fever, high temperature, running hot',
    'G02': 'chills, shivering, cold sweats',
    'G03': 'nausea, vomiting, cannot keep food down',
    'G04': 'fatigue, weakness, exhaustion, always tired',
    'G05': 'headache, dizziness, room spinning',
    'G06': 'swelling of face, legs, or feet',
    'G07': 'pain or burning when urinating',
    'G08': 'abnormal or smelly vaginal discharge',
    'G09': 'blurred or hazy vision',
    'G10': 'burning or sharp pain in the upper stomach or below the chest',
    'G11': 'shortness of breath, difficulty breathing',
    'G12': 'yellowish skin or eyes, jaundice',
    'G13': 'leg cramps, bone or calf pain',
    'G14': 'prolonged sadness, anxiety, hopelessness, frequent crying',
    'G15': 'constipation, bloating, no bowel movement',
}

vocab_lines = '\n'.join(
    f'- {sid}: {SYMPTOM_NAMES[sid]} (cues: {SYMPTOM_CUES[sid]})'
    for sid in sorted(SYMPTOM_NAMES)
)

SYSTEM_PROMPT = f"""You are a clinical information extraction assistant for maternal health visit notes.
Identify which symptoms from the fixed catalog below are explicitly mentioned in the transcript.

Symptom catalog:
{vocab_lines}

Rules:
- Respond with valid JSON only, no other text.
- Schema: {{"symptoms": [{{"symptom_id": "G01", "confidence": 0.95}}]}}
- Use only symptom_id values from the catalog.
- Report a symptom only if the transcript actually mentions it. Do not infer diagnoses or unstated symptoms.
- confidence is your certainty between 0 and 1 that the symptom is mentioned.
- If no catalog symptom is mentioned, respond with {{"symptoms": []}}."""

print(SYSTEM_PROMPT)

You are a clinical information extraction assistant for maternal health visit notes.
Identify which symptoms from the fixed catalog below are explicitly mentioned in the transcript.

Symptom catalog:
- G01: High fever (cues: fever, high temperature, running hot)
- G02: Chills (cues: chills, shivering, cold sweats)
- G03: Excessive nausea and vomiting (cues: nausea, vomiting, cannot keep food down)
- G04: Extreme fatigue (cues: fatigue, weakness, exhaustion, always tired)
- G05: Severe headache or dizziness (cues: headache, dizziness, room spinning)
- G06: Facial or leg swelling (cues: swelling of face, legs, or feet)
- G07: Painful urination (cues: pain or burning when urinating)
- G08: Abnormal vaginal discharge (cues: abnormal or smelly vaginal discharge)
- G09: Blurred vision (cues: blurred or hazy vision)
- G10: Epigastric pain (cues: burning or sharp pain in the upper stomach or below the chest)
- G11: Shortness of breath (cues: shortness of breath, difficulty breathing)
- G12: Ja

## Keyword Baseline Extractor

A transparent lexical matcher over the same closed vocabulary. It is the automatic fallback on CPU runtimes and the reference point for judging how much the LLM adds on top of simple keyword matching.

In [ ]:
"""Keyword baseline: case insensitive phrase matching per catalog symptom."""

import re

KEYWORD_PATTERNS = {
    'G01': ['fever', 'high temperature'],
    'G02': ['chill', 'shiver'],
    'G03': ['vomit', 'nausea'],
    'G04': ['weak', 'tired', 'exhaust'],
    'G05': ['headache', 'dizzy', 'head hurts', 'spins'],
    'G06': ['swollen', 'swelling'],
    'G07': ['urinat', 'passing urine'],
    'G08': ['discharge'],
    'G09': ['blurry', 'blurred', 'cannot see clearly', 'looks hazy'],
    'G10': ['upper part of her stomach', 'below her chest', 'epigastric'],
    'G11': ['short of breath', 'struggles to breathe', 'difficulty breathing'],
    'G12': ['yellow', 'jaundice'],
    'G13': ['cramp', 'bones ache'],
    'G14': ['sad', 'anxious', 'hopeless', 'worries', 'cries'],
    'G15': ['bowel', 'constipation', 'bloated'],
}


def extract_baseline(transcript):
    """Return [(symptom_id, confidence)] using lexical matching."""
    text = transcript.lower()
    found = []
    for symptom_id, patterns in KEYWORD_PATTERNS.items():
        if any(re.search(re.escape(p), text) for p in patterns):
            found.append((symptom_id, 0.90))
    return found, False

## Qwen Model Loading

Qwen3-4B-Instruct-2507 is loaded with four bit NF4 quantization, which keeps it around three to four gigabytes of VRAM so it coexists comfortably with Whisper on a sixteen gigabyte T4.

In [6]:
"""Load the quantized Qwen model when running on the LLM path."""

llm_model = None
llm_tokenizer = None

if USE_LLM:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        quantization_config=quant_config,
        device_map='auto',
    )
    llm_model.eval()
    print(f'loaded {LLM_MODEL_ID}')
    print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
else:
    print('LLM path disabled, keyword baseline will be used')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

loaded Qwen/Qwen3-4B-Instruct-2507
VRAM allocated: 2.85 GB


## LLM Extraction and JSON Parsing

The response is parsed defensively: locate the JSON object, validate the schema, drop identifiers outside the catalog, and clamp confidences to the unit interval. A single retry with an explicit correction message handles malformed replies; a second failure marks the record for human validation instead of guessing.

In [7]:
"""LLM extraction with strict parsing and a single retry on malformed output."""

JSON_RETRY_MESSAGE = (
    'Your previous reply was not valid JSON matching the schema. '
    'Respond again with only the JSON object.'
)


def parse_extraction_response(text):
    """Parse the model reply into [(symptom_id, confidence)] or None on failure."""
    start, end = text.find('{'), text.rfind('}')
    if start == -1 or end <= start:
        return None
    try:
        payload = json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None
    items = payload.get('symptoms')
    if not isinstance(items, list):
        return None
    parsed = []
    for item in items:
        if not isinstance(item, dict):
            return None
        symptom_id = item.get('symptom_id')
        if symptom_id not in VALID_SYMPTOM_IDS:
            continue
        try:
            confidence = float(item.get('confidence', 0.0))
        except (TypeError, ValueError):
            return None
        parsed.append((symptom_id, float(np.clip(confidence, 0.0, 1.0))))
    return parsed


def generate_reply(messages):
    """Greedy generation for one chat exchange."""
    encoded = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True,
    ).to(llm_model.device)
    with torch.no_grad():
        output = llm_model.generate(
            **encoded,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    prompt_length = encoded['input_ids'].shape[1]
    return llm_tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)


def extract_llm(transcript):
    """Return ([(symptom_id, confidence)], parse_failed) for one transcript."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Transcript: {transcript}'},
    ]
    reply = generate_reply(messages)
    parsed = parse_extraction_response(reply)
    if parsed is None:
        messages.append({'role': 'assistant', 'content': reply})
        messages.append({'role': 'user', 'content': JSON_RETRY_MESSAGE})
        parsed = parse_extraction_response(generate_reply(messages))
    if parsed is None:
        return [], True
    return parsed, False

## Batch Extraction

The corpus is processed sequentially with periodic progress reports. The environment variable MATERNALINK_EXTRACTION_LIMIT caps how many records go through the LLM; any remainder falls back to the keyword baseline so every record always carries an extraction result.

In [8]:
"""Run the extraction engine over the full anamnesis corpus."""

import time

EXTRACTION_LIMIT = int(os.environ.get('MATERNALINK_EXTRACTION_LIMIT', '0'))

llm_budget = EXTRACTION_LIMIT if (USE_LLM and EXTRACTION_LIMIT > 0) else len(anamnesis_records)

raw_rows = []
started = time.time()
for i, record in enumerate(anamnesis_records.itertuples()):
    if USE_LLM and i < llm_budget:
        symptoms, parse_failed = extract_llm(record.transcript)
        model_used = f'{LLM_MODEL_ID}-4bit'
    else:
        symptoms, parse_failed = extract_baseline(record.transcript)
        model_used = 'keyword-baseline'
    raw_rows.append({
        'anamnesis_id': record.anamnesis_id,
        'facility_id': record.facility_id,
        'period': record.period,
        'extracted_symptoms': symptoms,
        'parse_failed': parse_failed,
        'extraction_model': model_used,
    })
    if (i + 1) % 100 == 0:
        rate = (i + 1) / (time.time() - started)
        print(f'{i + 1}/{len(anamnesis_records)} records ({rate:.1f} rec/s)')

extraction_raw = pd.DataFrame(raw_rows)
elapsed = time.time() - started
print(f'completed {len(extraction_raw)} records in {elapsed / 60:.1f} min')

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


100/813 records (0.2 rec/s)
200/813 records (0.2 rec/s)
300/813 records (0.2 rec/s)
400/813 records (0.2 rec/s)
500/813 records (0.2 rec/s)
600/813 records (0.2 rec/s)
700/813 records (0.2 rec/s)
800/813 records (0.2 rec/s)
completed 813 records in 63.1 min


# Confidence and Human in the Loop

Records whose minimum symptom confidence falls below 0.85, that yield no symptoms at all, or that failed parsing are flagged for midwife validation. Because no midwife is available in this prototype, validation is simulated by substituting the ground truth labels for flagged records, exactly mirroring the production workflow where the midwife confirms or corrects the extraction before it enters the forecasting pipeline. Unflagged records keep the model output untouched.

In [9]:
"""Apply confidence thresholding and the simulated validation step."""

HITL_THRESHOLD = 0.85

ground_truth_map = anamnesis_ground_truth.set_index('anamnesis_id')['true_symptoms'].map(json.loads).to_dict()

result_rows = []
for row in extraction_raw.itertuples():
    confidences = [c for _, c in row.extracted_symptoms]
    min_confidence = min(confidences) if confidences else 0.0
    hitl_flag = bool(row.parse_failed or min_confidence < HITL_THRESHOLD)
    if hitl_flag:
        validated = sorted(ground_truth_map[row.anamnesis_id])
    else:
        validated = sorted(s for s, _ in row.extracted_symptoms)
    result_rows.append({
        'anamnesis_id': row.anamnesis_id,
        'facility_id': row.facility_id,
        'period': row.period,
        'extracted_symptoms': json.dumps([
            {'symptom_id': s, 'confidence': round(c, 3)} for s, c in row.extracted_symptoms
        ]),
        'min_confidence': round(min_confidence, 3),
        'hitl_flag': hitl_flag,
        'validated_symptoms': json.dumps(validated),
        'extraction_model': row.extraction_model,
    })

l0_extraction_results = pd.DataFrame(result_rows)
hitl_rate = l0_extraction_results['hitl_flag'].mean()
print(f'HITL flag rate: {hitl_rate:.1%}')
l0_extraction_results.head(3)

HITL flag rate: 2.2%


,anamnesis_id,facility_id,period,extracted_symptoms,min_confidence,hitl_flag,validated_symptoms,extraction_model
0,ANM-000001,PKM-001,2025-01-01,"[{""symptom_id"": ""G01"", ""confidence"": 0.95}, {""...",0.95,False,"[""G01"", ""G02"", ""G03"", ""G04""]",Qwen/Qwen3-4B-Instruct-2507-4bit
1,ANM-000002,PKM-001,2025-01-01,"[{""symptom_id"": ""G04"", ""confidence"": 0.95}, {""...",0.95,False,"[""G02"", ""G04""]",Qwen/Qwen3-4B-Instruct-2507-4bit
2,ANM-000003,PKM-001,2025-01-01,"[{""symptom_id"": ""G04"", ""confidence"": 0.98}]",0.98,False,"[""G04""]",Qwen/Qwen3-4B-Instruct-2507-4bit


# Probabilistic Condition Mapping

## Bayesian Scoring Function

For a validated symptom set S the unnormalized score of condition c is

prior_prevalence(c) multiplied by the product over symptoms g in S of likelihood(g given c), where the likelihood is inflated by no_lab_weight when the facility lacks a laboratory and replaced by a small epsilon when the knowledge base has no entry for the pair. Scores are normalized into a posterior across the sixteen conditions, and any condition whose posterior reaches 0.25 counts as indicated. The per symptom evidence terms are stored alongside each posterior so every indication remains auditable.

In [10]:
"""Bayesian condition scoring against the probabilistic knowledge base."""

EPSILON = 0.02
INDICATION_THRESHOLD = 0.25

KB_LOOKUP = {
    (row.symptom_id, row.condition_id): (row.likelihood, row.no_lab_weight)
    for row in kb_symptom_condition.itertuples()
}
CONDITION_PRIORS = conditions.set_index('condition_id')['prior_prevalence'].to_dict()


def condition_posteriors(symptoms, has_lab):
    """Return (posterior dict, evidence dict) for a validated symptom set."""
    scores = {}
    evidence = {}
    for condition_id, prior in CONDITION_PRIORS.items():
        score = prior
        terms = {}
        for symptom_id in symptoms:
            entry = KB_LOOKUP.get((symptom_id, condition_id))
            if entry is None:
                likelihood = EPSILON
            else:
                likelihood = entry[0] * (entry[1] if not has_lab else 1.0)
            score *= likelihood
            terms[symptom_id] = round(likelihood, 3)
        scores[condition_id] = score
        evidence[condition_id] = terms
    total = sum(scores.values())
    if total <= 0:
        uniform = 1.0 / len(scores)
        return {c: uniform for c in scores}, evidence
    return {c: s / total for c, s in scores.items()}, evidence

## Per Anamnesis Condition Posterior

In [11]:
"""Score every anamnesis record and keep the full posterior for auditability."""

posterior_rows = []
for row in l0_extraction_results.itertuples():
    symptoms = json.loads(row.validated_symptoms)
    posteriors, evidence = condition_posteriors(symptoms, HAS_LAB[row.facility_id])
    for condition_id, posterior in posteriors.items():
        if posterior < 0.01:
            continue
        posterior_rows.append({
            'anamnesis_id': row.anamnesis_id,
            'facility_id': row.facility_id,
            'period': row.period,
            'condition_id': condition_id,
            'posterior': round(posterior, 4),
            'indicated': posterior >= INDICATION_THRESHOLD,
            'evidence': json.dumps(evidence[condition_id]),
        })

l0_condition_posteriors = pd.DataFrame(posterior_rows)
indicated_share = l0_condition_posteriors.groupby('anamnesis_id')['indicated'].any().mean()
print(f'{len(l0_condition_posteriors)} posterior rows, {indicated_share:.1%} of records have at least one indicated condition')
l0_condition_posteriors.head()

4797 posterior rows, 100.0% of records have at least one indicated condition


,anamnesis_id,facility_id,period,condition_id,posterior,indicated,evidence
0,ANM-000001,PKM-001,2025-01-01,K01,0.9999,True,"{""G01"": 1.105, ""G02"": 0.975, ""G03"": 0.39, ""G04..."
1,ANM-000002,PKM-001,2025-01-01,K01,0.9200,True,"{""G02"": 0.975, ""G04"": 0.52}"
2,ANM-000002,PKM-001,2025-01-01,K02,0.0679,False,"{""G02"": 0.02, ""G04"": 1.04}"
3,ANM-000003,PKM-001,2025-01-01,K01,0.1909,False,"{""G04"": 0.52}"
4,ANM-000003,PKM-001,2025-01-01,K02,0.6872,True,"{""G04"": 1.04}"


# Aggregation to Layer 1 Contract

The contract table sums indicated posteriors into soft case counts per facility, period, and condition, then merges them with the manual diagnosis stream. The two streams are complementary by construction: anamnesis records originate from cases that never received a formal diagnosis, so adding them does not double count. The confidence level reflects how much of the anamnesis evidence passed extraction with high confidence or midwife validation.

In [12]:
"""Build l0_condition_estimates, the table Layer 1 joins on facility and period."""

record_confidence = l0_extraction_results.assign(
    record_confidence=lambda df: np.where(df['hitl_flag'], 1.0, df['min_confidence']),
)[['anamnesis_id', 'record_confidence']]

indicated = (
    l0_condition_posteriors[l0_condition_posteriors['indicated']]
    .merge(record_confidence, on='anamnesis_id')
)
anamnesis_agg = (
    indicated.groupby(['facility_id', 'period', 'condition_id'])
    .agg(
        anamnesis_indicated_cases=('posterior', 'sum'),
        mean_record_confidence=('record_confidence', 'mean'),
    )
    .reset_index()
)

manual_agg = (
    diagnoses_monthly.groupby(['facility_id', 'period', 'condition_id'])['case_count']
    .sum()
    .reset_index()
    .rename(columns={'case_count': 'manual_cases'})
)

l0_condition_estimates = manual_agg.merge(
    anamnesis_agg, on=['facility_id', 'period', 'condition_id'], how='outer',
)
l0_condition_estimates['manual_cases'] = l0_condition_estimates['manual_cases'].fillna(0).astype(int)
l0_condition_estimates['anamnesis_indicated_cases'] = (
    l0_condition_estimates['anamnesis_indicated_cases'].fillna(0.0).round(3)
)
l0_condition_estimates['estimated_total_cases'] = (
    l0_condition_estimates['manual_cases'] + l0_condition_estimates['anamnesis_indicated_cases']
).round(3)


def confidence_level(row):
    """High when purely manual or well validated, low when extraction was shaky."""
    if row['anamnesis_indicated_cases'] == 0:
        return 'high'
    if row['mean_record_confidence'] > 0.8:
        return 'high'
    if row['mean_record_confidence'] >= 0.5:
        return 'medium'
    return 'low'


l0_condition_estimates['confidence_level'] = l0_condition_estimates.apply(confidence_level, axis=1)
l0_condition_estimates = (
    l0_condition_estimates
    .drop(columns=['mean_record_confidence'])
    .sort_values(['facility_id', 'period', 'condition_id'])
    .reset_index(drop=True)
)

print(f'{len(l0_condition_estimates)} contract rows')
l0_condition_estimates.head()

6406 contract rows


,facility_id,period,condition_id,manual_cases,anamnesis_indicated_cases,estimated_total_cases,confidence_level
0,PKM-001,2023-01-01,K06,1,0.0,1.0,high
1,PKM-001,2023-01-01,K07,1,0.0,1.0,high
2,PKM-001,2023-01-01,K11,2,0.0,2.0,high
3,PKM-001,2023-01-01,K16,1,0.0,1.0,high
4,PKM-001,2023-02-01,K01,1,0.0,1.0,high


# Audio Demo Pipeline

A small end to end demonstration of the speech path: a handful of transcripts are synthesized to audio (or real recordings dropped into the audio_demo folder are used instead), transcribed back with faster-whisper, and pushed through the same extraction engine. This proves the full chain from voice to structured symptom output without making audio part of the bulk pipeline.

In [13]:
"""Synthesize demo audio if the folder is empty, then transcribe and extract."""

N_AUDIO_DEMO = 8

try:
    from faster_whisper import WhisperModel
    HAS_WHISPER = True
except ModuleNotFoundError:
    HAS_WHISPER = False

try:
    from gtts import gTTS
    HAS_GTTS = True
except ModuleNotFoundError:
    HAS_GTTS = False

audio_demo_results = None
if not HAS_WHISPER:
    print('faster-whisper not installed, skipping the audio demo on this runtime')
else:
    demo_records = anamnesis_records.sample(N_AUDIO_DEMO, random_state=SEED)
    existing = [f for f in os.listdir(DATA_DIRS['audio_demo']) if f.endswith(('.mp3', '.wav'))]
    if not existing and HAS_GTTS:
        for record in demo_records.itertuples():
            tts = gTTS(record.transcript, lang='en')
            tts.save(os.path.join(DATA_DIRS['audio_demo'], f'{record.anamnesis_id}.mp3'))
        existing = sorted(f for f in os.listdir(DATA_DIRS['audio_demo']) if f.endswith('.mp3'))
        print(f'synthesized {len(existing)} demo clips with gTTS')

    if not existing:
        print('no audio files available and gTTS missing, skipping transcription')
    else:
        whisper_device = 'cuda' if HAS_CUDA else 'cpu'
        whisper_compute = 'int8_float16' if HAS_CUDA else 'int8'
        stt_model = WhisperModel('small', device=whisper_device, compute_type=whisper_compute)

        """The four bit Qwen model and Whisper small fit together on a T4, so the
        demo uses the same extractor path as the bulk pipeline."""
        demo_extract = extract_llm if USE_LLM else extract_baseline
        demo_rows = []
        for filename in existing:
            segments, _ = stt_model.transcribe(os.path.join(DATA_DIRS['audio_demo'], filename))
            stt_text = ' '.join(seg.text.strip() for seg in segments)
            symptoms, _ = demo_extract(stt_text)
            demo_rows.append({
                'audio_file': filename,
                'stt_model': 'faster-whisper-small',
                'stt_transcript': stt_text,
                'extracted_symptoms': json.dumps([
                    {'symptom_id': s, 'confidence': round(c, 3)} for s, c in symptoms
                ]),
            })
        audio_demo_results = pd.DataFrame(demo_rows)
        audio_demo_results.to_csv(
            os.path.join(DATA_DIRS['eval'], 'audio_demo_results.csv'), index=False,
        )
        print(f'audio demo complete for {len(audio_demo_results)} clips')
        display(audio_demo_results.head())

synthesized 8 demo clips with gTTS


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


audio demo complete for 8 clips


,audio_file,stt_model,stt_transcript,extracted_symptoms
0,ANM-000199.mp3,faster-whisper-small,During the visit she mentions that she struggl...,"[{""symptom_id"": ""G11"", ""confidence"": 0.95}, {""..."
1,ANM-000228.mp3,faster-whisper-small,"During the visit, she mentions that it burns a...","[{""symptom_id"": ""G07"", ""confidence"": 0.95}, {""..."
2,ANM-000248.mp3,faster-whisper-small,She complains that her head hurts badly and th...,"[{""symptom_id"": ""G05"", ""confidence"": 0.95}, {""..."
3,ANM-000292.mp3,faster-whisper-small,The patient says that she feels hopeless and w...,"[{""symptom_id"": ""G14"", ""confidence"": 0.95}]"
4,ANM-000295.mp3,faster-whisper-small,She complains that she vomits almost everythin...,"[{""symptom_id"": ""G03"", ""confidence"": 0.98}]"


# Persistence and Evaluation

## Write Layer 0 Output Tables

In [14]:
"""Persist extraction results, posteriors, and the Layer 1 contract table."""


def write_csv(df, folder, filename):
    out = df.copy()
    if 'period' in out.columns:
        out['period'] = pd.to_datetime(out['period']).dt.strftime('%Y-%m-%d')
    path = os.path.join(DATA_DIRS[folder], filename)
    out.to_csv(path, index=False)
    print(f'wrote {len(out):>7} rows -> {path}')


write_csv(l0_extraction_results, 'layer0_output', 'l0_extraction_results.csv')
write_csv(l0_condition_posteriors, 'layer0_output', 'l0_condition_posteriors.csv')
write_csv(l0_condition_estimates, 'layer0_output', 'l0_condition_estimates.csv')

wrote     813 rows -> /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton/data/layer0_output/l0_extraction_results.csv
wrote    4797 rows -> /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton/data/layer0_output/l0_condition_posteriors.csv
wrote    6406 rows -> /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton/data/layer0_output/l0_condition_estimates.csv


## Extraction Accuracy vs Ground Truth

Metrics are computed on the raw model output before any human in the loop substitution, which keeps the evaluation honest about what the extractor alone achieves. Reported figures cover micro precision, recall and F1, per symptom breakdowns, the false positive profile, and how often the true source condition lands in the top posterior ranks.

In [15]:
"""Multi label extraction metrics on raw output, before HITL substitution."""

eval_df = extraction_raw.merge(anamnesis_ground_truth, on='anamnesis_id')

tp = fp = fn = 0
per_symptom = {sid: {'tp': 0, 'fp': 0, 'fn': 0} for sid in sorted(VALID_SYMPTOM_IDS)}
for row in eval_df.itertuples():
    predicted = {s for s, _ in row.extracted_symptoms}
    actual = set(json.loads(row.true_symptoms))
    for sid in predicted & actual:
        tp += 1
        per_symptom[sid]['tp'] += 1
    for sid in predicted - actual:
        fp += 1
        per_symptom[sid]['fp'] += 1
    for sid in actual - predicted:
        fn += 1
        per_symptom[sid]['fn'] += 1

precision = tp / max(1, tp + fp)
recall = tp / max(1, tp + fn)
f1 = 2 * precision * recall / max(1e-9, precision + recall)

per_symptom_table = pd.DataFrame([
    {
        'symptom_id': sid,
        'symptom_name': SYMPTOM_NAMES[sid],
        'precision': counts['tp'] / max(1, counts['tp'] + counts['fp']),
        'recall': counts['tp'] / max(1, counts['tp'] + counts['fn']),
        'support': counts['tp'] + counts['fn'],
    }
    for sid, counts in per_symptom.items()
]).round(3)

print(f'micro precision {precision:.3f}, recall {recall:.3f}, F1 {f1:.3f}')
per_symptom_table

micro precision 0.980, recall 0.999, F1 0.989


,symptom_id,symptom_name,precision,recall,support
0,G01,High fever,0.995,1.000,202
1,G02,Chills,0.932,1.000,124
2,G03,Excessive nausea and vomiting,0.990,1.000,100
3,G04,Extreme fatigue,0.987,0.996,232
4,G05,Severe headache or dizziness,0.980,1.000,145
5,G06,Facial or leg swelling,0.981,1.000,52
6,G07,Painful urination,1.000,1.000,64
7,G08,Abnormal vaginal discharge,1.000,1.000,44
8,G09,Blurred vision,0.980,1.000,48
9,G10,Epigastric pain,0.971,1.000,102


In [16]:
"""Condition ranking metrics: does the true source condition surface on top."""

top1_hits = top2_hits = 0
for row in eval_df.itertuples():
    symptoms = sorted({s for s, _ in row.extracted_symptoms})
    posteriors, _ = condition_posteriors(symptoms, HAS_LAB[row.facility_id])
    ranking = sorted(posteriors, key=posteriors.get, reverse=True)
    if row.source_condition == ranking[0]:
        top1_hits += 1
    if row.source_condition in ranking[:2]:
        top2_hits += 1

top1 = top1_hits / len(eval_df)
top2 = top2_hits / len(eval_df)
print(f'source condition in top 1 posterior: {top1:.1%}')
print(f'source condition in top 2 posterior: {top2:.1%}')

source condition in top 1 posterior: 85.1%
source condition in top 2 posterior: 95.4%


In [17]:
"""Persist the evaluation summary and assert the pipeline quality gates."""

eval_summary = {
    'extraction_model': sorted(extraction_raw['extraction_model'].unique()),
    'n_records': len(eval_df),
    'micro_precision': round(precision, 3),
    'micro_recall': round(recall, 3),
    'micro_f1': round(f1, 3),
    'hitl_flag_rate': round(float(hitl_rate), 3),
    'source_condition_top1': round(top1, 3),
    'source_condition_top2': round(top2, 3),
}
with open(os.path.join(DATA_DIRS['eval'], 'l0_eval_summary.json'), 'w') as f:
    json.dump(eval_summary, f, indent=2)

assert f1 > 0.6, 'extraction F1 below the minimum quality gate'
assert top2 > 0.5, 'condition ranking below the minimum quality gate'

for key, value in eval_summary.items():
    print(f'{key:>25}: {value}')

         extraction_model: ['Qwen/Qwen3-4B-Instruct-2507-4bit']
                n_records: 813
          micro_precision: 0.98
             micro_recall: 0.999
                 micro_f1: 0.989
           hitl_flag_rate: 0.022
    source_condition_top1: 0.851
    source_condition_top2: 0.954
